[![Ouvrir dans Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/laleye/IA-Training-participants/blob/main/module1/commun/notebooks/S0_verification_environnement.ipynb)

# S0 — Vérification de l'environnement

**À faire avant la session 1** · Durée : ≈ 5 minutes · Ticket `M1-G03`

Ce notebook vérifie, en une seule exécution, tout ce dont vous aurez besoin pendant les ateliers :
GPU, bibliothèques, lecture audio, accès à Hugging Face, données du cours, reconnaissance vocale.

**Mode d'emploi**
1. Sur Colab : menu **Exécution → Modifier le type d'exécution → GPU T4**, puis **Enregistrer**.
2. Menu **Exécution → Tout exécuter** (ou `Ctrl+F9`).
3. Lisez le **rapport final** en bas de page : ✅ tout va bien · ⚠️ à corriger avant la session concernée · ❌ bloquant.

Aucun code à écrire. En cas de problème, consultez la section *Dépannage* du guide d'installation.

In [ ]:
# ⏳ Installation (≈ 1 min sur Colab, instantané ensuite). Ne pas modifier.
import importlib.util, subprocess, sys
if importlib.util.find_spec("m1utils") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "m1utils[all] @ git+https://github.com/laleye/IA-Training-participants.git"], check=True)
import m1utils
INFO = m1utils.setup_notebook()

In [ ]:
import time
import traceback

RAPPORT = []  # (vérification, statut, détail, action)


def verifier(nom, fonction, action_si_echec, bloquant=True):
    """Exécute une vérification sans interrompre les suivantes."""
    debut = time.perf_counter()
    try:
        detail = fonction()
        RAPPORT.append((nom, "✅", f"{detail} ({time.perf_counter() - debut:.1f} s)", ""))
    except Exception as exc:
        statut = "❌" if bloquant else "⚠️"
        RAPPORT.append((nom, statut, f"{type(exc).__name__} : {str(exc)[:150]}", action_si_echec))
        traceback.print_exc(limit=1)
    print(RAPPORT[-1][1], nom, "—", RAPPORT[-1][2])

## 1. Versions des bibliothèques

In [ ]:
def versions():
    import datasets, librosa, torch, transformers
    return (f"Python {sys.version.split()[0]} · m1utils {m1utils.__version__} · torch {torch.__version__} · "
            f"transformers {transformers.__version__} · datasets {datasets.__version__} · librosa {librosa.__version__}")

verifier("Bibliothèques", versions, "Relancer la cellule d'installation ; sinon Exécution → Redémarrer la session.")

## 2. Carte graphique (GPU)

In [ ]:
def gpu():
    import torch
    if not torch.cuda.is_available():
        raise RuntimeError("aucun GPU détecté")
    nom = torch.cuda.get_device_name(0)
    memoire = torch.cuda.get_device_properties(0).total_memory / 1e9
    return f"{nom}, {memoire:.0f} Go"

verifier("GPU", gpu, "Colab : Exécution → Modifier le type d'exécution → GPU T4. Sans GPU, les ateliers S3/S4 seront lents.",
         bloquant=False)

## 3. Lecture et écriture audio

In [ ]:
from m1utils.audio import load_audio, play, save_audio, synth_tone

def audio_io():
    y = synth_tone(440, 1.0)
    chemin = save_audio("/tmp/m1_test.wav", y)
    z, sr = load_audio(chemin)
    assert sr == 16_000 and abs(len(z) - 16_000) <= 1
    return "écriture + lecture WAV 16 kHz OK"

verifier("Audio (lecture/écriture)", audio_io, "Réinstaller soundfile : pip install soundfile")
play(synth_tone(440, 1.0), label="🔊 Vous devez entendre un « la » (440 Hz) d'une seconde :")

## 4. Accès au Hub Hugging Face (nécessaire en session 4)

Sur Colab : icône 🔑 **Secrets** (barre de gauche) → ajouter `HF_TOKEN` avec votre jeton *write*,
et activer « Accès au notebook ». Ne collez **jamais** votre jeton dans une cellule.

In [ ]:
def hub():
    import os
    from huggingface_hub import whoami
    token = os.environ.get("HF_TOKEN")
    if token is None and INFO["colab"]:
        from google.colab import userdata
        token = userdata.get("HF_TOKEN")
    if not token:
        raise RuntimeError("jeton HF_TOKEN absent")
    return f"connecté en tant que {whoami(token=token)['name']}"

verifier("Hugging Face (jeton)", hub, "Créer un jeton write sur huggingface.co/settings/tokens et l'ajouter aux Secrets Colab (HF_TOKEN).",
         bloquant=False)

## 5. Données du cours

In [ ]:
from m1utils.data import load_manifest

def donnees():
    df = load_manifest("demo_s1")
    return f"pack demo_s1 : {len(df)} énoncés ({', '.join(f'{l}={n}' for l, n in df.lang.value_counts().items())})"

verifier("Données (pack demo_s1)", donnees, "Vérifier la connexion internet puis relancer ; voir le guide d'installation.")

## 6. Reconnaissance vocale : premier essai
On transcrit un énoncé en fongbe avec **Simba-H**, un modèle léger spécialisé dans les langues africaines.

In [ ]:
from m1utils.asr import load_asr, unload

def asr():
    df = load_manifest("demo_s1")
    ligne = df[df.lang == "fon"].iloc[0]
    modele = load_asr("simba-h")
    resultat = modele(ligne.abspath)
    unload(modele)
    return f"référence « {ligne.text} » → modèle « {resultat.text} » (calcul : {resultat.latency_s:.2f} s)"

verifier("Reconnaissance vocale (Simba-H)", asr, "Vérifier l'accès à huggingface.co ; relancer la cellule.")

## 7. Datasets Hugging Face (format audio)

In [ ]:
def datasets_audio():
    from datasets import Audio, Dataset
    from m1utils.audio import decode_bytes
    data = open("/tmp/m1_test.wav", "rb").read()
    ds = Dataset.from_dict({"audio": [{"bytes": data, "path": "test.wav"}]}).cast_column("audio", Audio(decode=False))
    y, sr = decode_bytes(ds[0]["audio"]["bytes"])
    return f"Dataset audio OK ({len(y) / sr:.1f} s)"

verifier("Datasets (audio)", datasets_audio, "Mettre à jour : pip install -U datasets")

## 8. Micro (optionnel, Colab uniquement)
Passez `TESTER_MICRO = True`, exécutez, autorisez le micro dans le navigateur et dites quelques mots.
Sans micro, vous pourrez utiliser les enregistrements de votre smartphone.

In [ ]:
TESTER_MICRO = False

if TESTER_MICRO:
    from m1utils.audio import record
    def micro():
        y, sr = record(seconds=3)
        play(y, sr, label="Votre enregistrement :")
        return f"{len(y) / sr:.1f} s enregistrées"
    verifier("Micro", micro, "Autoriser le micro dans le navigateur (icône 🔒 de la barre d'adresse).", bloquant=False)

## 📋 Rapport final

In [ ]:
import pandas as pd
from IPython.display import Markdown, display

rapport = pd.DataFrame(RAPPORT, columns=["Vérification", "Statut", "Détail", "Action si problème"])
display(rapport.style.hide(axis="index"))
bloquants = (rapport["Statut"] == "❌").sum()
alertes = (rapport["Statut"] == "⚠️").sum()
if bloquants:
    display(Markdown(f"### ❌ {bloquants} problème(s) bloquant(s)\nSuivez la colonne « Action », puis relancez tout. "
                     "Si cela persiste, écrivez sur le canal de support **avant** la session 1 en joignant une capture de ce tableau."))
elif alertes:
    display(Markdown(f"### ⚠️ Prêt pour la session 1, avec {alertes} point(s) à régler d'ici la session 4\n"
                     "(GPU et jeton Hugging Face)."))
else:
    display(Markdown("### ✅ Prêt pour la session 1 ! À bientôt."))